API Key & Keamanan

In [1]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "PGABL-legal-chatbot-finetune"
os.environ["WANDB_LOG_MODEL"] = "false"
print("Secrets termuat ke environment variables:", all(k in os.environ for k in ["HF_TOKEN", "WANDB_API_KEY"]))

Secrets termuat ke environment variables: True


Instalasi

In [2]:
# Mengunci seluruh versi dependensi utama agar 100% kompatibel dengan Unsloth di Kaggle
!pip install -q --upgrade unsloth unsloth_zoo
!pip install -q "huggingface-hub<2.0.0"
!pip install -q "datasets<5.0.0,>=3.4.1"
!pip install -q --upgrade wandb
!pip list 2>/dev/null | grep -i -E "^(unsloth|trl|peft|datasets|transformers|bitsandbytes|torch|huggingface-hub) "


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 64.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 66.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 68.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 82.8 MB/s eta 0:00:00:00:0100:0

Dataset & Chat Template (Llama-3.1)

In [3]:
import random
from datasets import load_dataset

SEED = 3407
SYSTEM_PROMPT = "Kamu adalah asisten AI yang membantu menjawab pertanyaan pengguna berdasarkan konteks yang diberikan."

raw_dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")
print("Total baris:", len(raw_dataset))
print("Nama kolom asli dataset:", raw_dataset.column_names)

print("\n=== SEBELUM mapping: dataset[0] ===")
print(raw_dataset[0])

# Subset agar training 800 steps tetap efisien di Kaggle
N_SAMPLES = min(20000, len(raw_dataset))
raw_dataset = raw_dataset.shuffle(seed=SEED).select(range(N_SAMPLES))

def to_llama3_chat(example):
    # Mengantisipasi jika nama kolom menggunakan huruf besar/kecil (Instruction vs instruction)
    key_map = {k.lower(): k for k in example.keys()}
    
    inst_key = key_map.get("instruction")
    input_key = key_map.get("input")
    out_key = key_map.get("output")
    
    instruction = str(example[inst_key] if inst_key else "").strip()
    extra_input = str(example[input_key] if input_key else "").strip()
    output = str(example[out_key] if out_key else "").strip()
    
    user_msg = instruction if not extra_input else f"{instruction}\n\n{extra_input}"
    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        "Cutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_msg}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{output}<|eot_id|>"
    )
    return {"text": text}

mapped_dataset = raw_dataset.map(
    to_llama3_chat,
    remove_columns=raw_dataset.column_names,
    num_proc=2,
    desc="Mapping ke Llama-3 chat template",
)

print("\n=== SESUDAH mapping: dataset[0]['text'] ===")
print(mapped_dataset[0]["text"])

# Split eksplisit train / eval (Skilled)
split = mapped_dataset.train_test_split(test_size=0.01, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"\ntrain_dataset: {len(train_dataset)} baris | eval_dataset: {len(eval_dataset)} baris")


README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv: reconstructing file:   0%|          |  0.00B / 41.4MB            

alpaca-gpt4-indonesia.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Total baris: 49969
Nama kolom asli dataset: ['Unnamed: 0', 'input', 'output']

=== SEBELUM mapping: dataset[0] ===
{'Unnamed: 0': 1, 'input': 'Saranlah slogan untuk kampanye daur ulang\n', 'output': '1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."\n2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."\n3. "Ubah sampahmu menjadi harta karun - Daur ulang!"\n4. "Daur ulang untuk siklus kehidupan."\n5. "Simpan sumber daya, daur ulang lebih banyak."'}


Mapping ke Llama-3 chat template (num_proc=2):   0%|          | 0/20000 [00:00<?, ? examples/s]


=== SESUDAH mapping: dataset[0]['text'] ===
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Kamu adalah asisten AI yang membantu menjawab pertanyaan pengguna berdasarkan konteks yang diberikan.<|eot_id|><|start_header_id|>user<|end_header_id|>



Apa tiga prinsip terpenting yang harus dipertimbangkan saat membuat infografis?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Saat membuat infografis, ada tiga prinsip penting yang harus dipertimbangkan:

1. **Ketegasan**: Infografis harus menyajikan informasi dengan cara yang jelas dan singkat. Ini harus mudah dipahami dan diinterpretasikan, bahkan untuk seseorang yang tidak terbiasa dengan topik tersebut. Untuk mencapai ketegasan, penting untuk menggunakan bahasa yang sederhana, bantuan visual seperti ikon atau ilustrasi, dan layout yang terorganisir dengan baik.

2. **Relevansi**: Informasi dalam infografis harus relevan dengan audiens yang ditujukan. Pe

Model 4-bit QLoRA + LoRA Adapter

In [4]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 1024

# 4-bit NF4 + double quantization (Unsloth mengaktifkan bnb_4bit_use_double_quant=True secara internal)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",  # alternatif: "unsloth/Meta-Llama-3.1-8B-Instruct"
    max_seq_length=max_seq_length,
    dtype=None,            # auto: fp16 di T4/P100, bf16 jika didukung
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)

# Bukti konfigurasi kuantisasi untuk reviewer
try:
    print("Quantization config:", model.config.quantization_config)
except Exception as e:
    print("Tidak bisa membaca quantization_config:", e)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # Multi-Head Attention
        "gate_proj", "up_proj", "down_proj",      # Feed Forward Network
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
)
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.7: Fast Llama patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


Quantization config: {'bnb_4bit_compute_dtype': torch.float16, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.1.mlp'], 'llm_int8_threshold': 6.0, 'load_in_4bit': True, 'load_in_8bit': False, 'quant_method': 'bitsandbytes'}


Unsloth 2026.10.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


SFTTrainer + Eval + WandB

eksperimen 1

In [5]:
import inspect, wandb
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

wandb.login(key=os.environ["WANDB_API_KEY"])

# === Eksperimen (Skilled): jalankan >=2x dengan CONFIG berbeda & run_name berbeda ===
CONFIG = {
    "run_name": "exp1-lr2e-4-r16",
    "learning_rate": 2e-4,
    "max_steps": 800,                 # minimal 800
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8, # effective batch = 16
    "weight_decay": 0.01,
    "warmup_steps": 40,
    "lr_scheduler_type": "cosine",
}

# 'evaluation_strategy' diganti nama menjadi 'eval_strategy' di transformers >= 4.46
eval_key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"

training_args = SFTConfig(
    output_dir=f"outputs/{CONFIG['run_name']}",
    run_name=CONFIG["run_name"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    dataset_num_proc=2,
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    max_steps=CONFIG["max_steps"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=CONFIG["warmup_steps"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_strategy="steps",
    save_steps=400,
    save_total_limit=2,
    seed=SEED,
    report_to="wandb",
    **{eval_key: "steps"},
    eval_steps=100,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

torch.cuda.empty_cache()
trainer_stats = trainer.train()
print(trainer_stats)
print("Eval akhir:", trainer.evaluate())
print(f"Peak VRAM: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")
wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: putraharifin (putraharifin-universitas-sumatera-utara) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/19800 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

...

Unsloth: Not an error, but `num_items_in_batch` could not be counted for LlamaForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs/exp1-lr2e-4-r16/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/exp1-lr2e-4-r16/checkpoint-800/tokenizer_config.json.


TrainOutput(global_step=800, training_loss=1.0408525848388672, metrics={'train_runtime': 5460.55, 'train_samples_per_second': 2.344, 'train_steps_per_second': 0.147, 'total_flos': 6.438192939035443e+16, 'train_loss': 1.0408525848388672, 'epoch': 0.6464646464646465})


Training Loss,Validation Loss,Step
0.973439,0.963634,800


Eval akhir: {'eval_loss': 0.9636343121528625}
Peak VRAM: 4.13 GB


eval/loss,█▆▄▃▂▁▁▁▁
eval/runtime,█▁▁▁▂▁▂▁▁
eval/samples_per_second,▁█▇█▇█▇██
eval/steps_per_second,▁█▇█▇█▇██
train/epoch,▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▇▃▂█▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▂▂▂▂▂▂▂▂▃▂▂▂▃▂▂
train/learning_rate,▃▄▆██████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁▁▁
train/loss,█▃▂▂▁▂▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.96363
eval/runtime,30.0503


eksperimen 2

In [5]:
import inspect, wandb
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

wandb.login(key=os.environ["WANDB_API_KEY"])

# === Eksperimen (Skilled): jalankan >=2x dengan CONFIG berbeda & run_name berbeda ===
CONFIG = {
    "run_name": "exp2-lr1e-4-r16",
    "learning_rate": 1e-4,
    "max_steps": 800,                 # minimal 800
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8, # effective batch = 16
    "weight_decay": 0.01,
    "warmup_steps": 40,
    "lr_scheduler_type": "cosine",
}

# 'evaluation_strategy' diganti nama menjadi 'eval_strategy' di transformers >= 4.46
eval_key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"

training_args = SFTConfig(
    output_dir=f"outputs/{CONFIG['run_name']}",
    run_name=CONFIG["run_name"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    dataset_num_proc=2,
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    max_steps=CONFIG["max_steps"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=CONFIG["warmup_steps"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_strategy="steps",
    save_steps=400,
    save_total_limit=2,
    seed=SEED,
    report_to="wandb",
    **{eval_key: "steps"},
    eval_steps=100,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

torch.cuda.empty_cache()
trainer_stats = trainer.train()
print(trainer_stats)
print("Eval akhir:", trainer.evaluate())
print(f"Peak VRAM: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")
wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: putraharifin (putraharifin-universitas-sumatera-utara) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/19800 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

...

Unsloth: Not an error, but `num_items_in_batch` could not be counted for LlamaForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs/exp2-lr1e-4-r16/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/exp2-lr1e-4-r16/checkpoint-800/tokenizer_config.json.


TrainOutput(global_step=800, training_loss=1.0624882805347442, metrics={'train_runtime': 5728.5513, 'train_samples_per_second': 2.234, 'train_steps_per_second': 0.14, 'total_flos': 6.438192939035443e+16, 'train_loss': 1.0624882805347442, 'epoch': 0.6464646464646465})


Training Loss,Validation Loss,Step
0.991250,0.981082,800


Eval akhir: {'eval_loss': 0.9810824394226074}
Peak VRAM: 4.13 GB


eval/loss,█▄▃▂▂▁▁▁▁
eval/runtime,█▂▂▁▁▁▁▁▁
eval/samples_per_second,▁▇▇██████
eval/steps_per_second,▁▇▇█████▇
train/epoch,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/grad_norm,█▇▁▁▂▁▄▂▂▂▂▂▂▂▂▃▂▂▂▃▃▂▃▃▃▄▃▃▃▃▃▃▃▃▃▃▂▃▃▃
train/learning_rate,▄▆████████▇▇▇▆▆▆▆▅▄▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
train/loss,█▂▂▂▂▂▂▂▂▂▂▁▁▂▁▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁
eval/loss,0.98108
eval/runtime,31.3544


Push ke Hugging Face (merged_16bit)

In [6]:
HF_USERNAME = "putraharifin"                     
REPO_NAME = "llama-3.2-3b-legal-id-PGABL-putraharifin25" 
repo_id = f"{HF_USERNAME}/{REPO_NAME}"

model.push_to_hub_merged(
    repo_id,
    tokenizer,
    save_method="merged_16bit",
    token=os.environ["HF_TOKEN"],
    private=False,   # WAJIB public, repo private = submission ditolak
)

link = f"https://huggingface.co/{repo_id}"
with open("link_huggingface.txt", "w") as f:
    f.write(link + "\n")
print("Model terunggah:", link)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in putraharifin/llama-3.2-3b-legal-id-PGABL-putraharifin25/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:39<01:39, 99.53s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:20<00:00, 70.11s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/putraharifin/llama-3.2-3b-legal-id-PGABL-putraharifin25`
Model terunggah: https://huggingface.co/putraharifin/llama-3.2-3b-legal-id-PGABL-putraharifin25
